<a href="https://colab.research.google.com/github/machancejoy-max/colab-git-demo-JOY/blob/main/Assignment_8_PAAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset, random_split
import torchvision
import torchvision.transforms as transforms
import numpy as np

In [ ]:
# Define preprocessing steps for CIFAR-10 images
transform = transforms.Compose([
    transforms.ToTensor(),                      # Convert images to PyTorch tensors
    transforms.Normalize((0.5, 0.5, 0.5),       # Normalize each channel (R,G,B)
                         (0.5, 0.5, 0.5))
])

# Load the full CIFAR-10 training dataset
train_full = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Load the CIFAR-10 test dataset
test_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)



100%|██████████| 170M/170M [09:49<00:00, 289kB/s]


In [ ]:

# Convert labels to a NumPy array for easy indexing
train_labels = np.array(train_full.targets)

# indices for classes 0–4 (Subset A) and 5–9 (Subset B)
indices_A = np.where((train_labels >= 0) & (train_labels <= 4))[0]
indices_B = np.where((train_labels >= 5) & (train_labels <= 9))[0]


In [ ]:
def make_balanced_indices(indices, labels, class_list, per_class=3000):
    balanced = []
    for c in class_list:
        class_idx = indices[labels[indices] == c]   # all indices of class c
        class_idx = class_idx[:per_class]           # take first N samples
        balanced.extend(class_idx)
    return balanced

balanced_A = make_balanced_indices(indices_A, train_labels, [0,1,2,3,4], per_class=3000)
balanced_B = make_balanced_indices(indices_B, train_labels, [5,6,7,8,9], per_class=3000)

subsetA = Subset(train_full, balanced_A)
subsetB = Subset(train_full, balanced_B)


In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset, random_split
val_ratio = 0.2
val_size = int(len(subsetA) * val_ratio)
train_size = len(subsetA) - val_size

trainA, valA = random_split(subsetA, [train_size, val_size])


In [ ]:
batch_size = 64

train_loaderA = DataLoader(trainA, batch_size=batch_size, shuffle=True)
val_loaderA = DataLoader(valA, batch_size=batch_size, shuffle=False)


In [ ]:
import torch.nn as nn
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 32 filters
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 32x16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 64 filters
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 64x8x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)                  # 5 classes: 0–4
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(num_classes=5).to(device)


In [ ]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loaderA:
        images = images.to(device)
        labels = labels.to(device)

        # labels are 0–4 already, no remapping needed

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    # validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loaderA:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


Epoch [1/10] Train Loss: 1.0345, Train Acc: 0.5732 Val Loss: 0.9064, Val Acc: 0.6477
Epoch [2/10] Train Loss: 0.8108, Train Acc: 0.6785 Val Loss: 0.7801, Val Acc: 0.6937
Epoch [3/10] Train Loss: 0.7083, Train Acc: 0.7248 Val Loss: 0.7386, Val Acc: 0.7110
Epoch [4/10] Train Loss: 0.6165, Train Acc: 0.7660 Val Loss: 0.6643, Val Acc: 0.7480
Epoch [5/10] Train Loss: 0.5470, Train Acc: 0.7887 Val Loss: 0.6543, Val Acc: 0.7507
Epoch [6/10] Train Loss: 0.4761, Train Acc: 0.8173 Val Loss: 0.6939, Val Acc: 0.7397
Epoch [7/10] Train Loss: 0.4121, Train Acc: 0.8441 Val Loss: 0.6648, Val Acc: 0.7623
Epoch [8/10] Train Loss: 0.3427, Train Acc: 0.8744 Val Loss: 0.6867, Val Acc: 0.7480
Epoch [9/10] Train Loss: 0.2664, Train Acc: 0.9037 Val Loss: 0.7532, Val Acc: 0.7590
Epoch [10/10] Train Loss: 0.2139, Train Acc: 0.9237 Val Loss: 0.8173, Val Acc: 0.7513


In [ ]:
subsetB_loader = DataLoader(subsetB, batch_size=64, shuffle=False)
#Evaluate Model A on Subset B

model.eval()   # Set model to evaluation mode

correct = 0
total = 0
loss_sum = 0
criterion = nn.CrossEntropyLoss()

with torch.no_grad():   # No gradients needed for evaluation
    for images, labels in subsetB_loader:
        images, labels = images.to(device), labels.to(device)
        # Remap labels from 5-9 to 0-4 for evaluation with model trained on 0-4
        labels = labels - 5

        outputs = model(images)             # Forward pass
        loss = criterion(outputs, labels)    # Compute loss
        loss_sum += loss.item()

        _, predicted = outputs.max(1)        # Predicted class
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

subsetB_acc = correct / total
subsetB_loss = loss_sum / len(subsetB_loader)

print(f"Performance on Subset B (unseen classes):")
print(f"Loss: {subsetB_loss:.4f} | Accuracy: {subsetB_acc:.4f}")

Performance on Subset B (unseen classes):
Loss: 6.5952 | Accuracy: 0.0873


In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset

# Simple evaluation helper
def evaluate(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    correct, total, loss_sum = 0, 0, 0.0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss_sum += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    acc = correct / total
    loss = loss_sum / len(loader)
    return acc, loss


In [ ]:
# Combine Subset A and Subset B into one dataset
combined_dataset = ConcatDataset([subsetA, subsetB])

# Split combined dataset into train and validation (e.g., 80/20)
combined_size = len(combined_dataset)
train_size = int(0.8 * combined_size)
val_size = combined_size - train_size

train_combined, val_combined = torch.utils.data.random_split(
    combined_dataset,
    [train_size, val_size]
)

train_combined_loader = DataLoader(train_combined, batch_size=64, shuffle=True)
val_combined_loader = DataLoader(val_combined, batch_size=64, shuffle=False)


In [ ]:
# New model (stateless retraining)
model_stateless = SimpleCNN(num_classes=10).to(device) # Change num_classes to 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_stateless.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):
    model_stateless.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_combined_loader:
        images, labels = images.to(device), labels.to(device)
        # Labels from combined_dataset can be 0-9, so they should be used as is with a 10-class model

        optimizer.zero_grad()
        outputs = model_stateless(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_combined_loader)
    train_acc = correct / total

    print(f"[Stateless] Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
stateless_val_acc, stateless_val_loss = evaluate(
    model_stateless, val_combined_loader, device
)

print(f"[Stateless] Combined Val Loss: {stateless_val_loss:.4f} | "
      f"Combined Val Acc: {stateless_val_acc:.4f}")

[Stateless] Epoch 1/10 | Train Loss: 1.5193 | Train Acc: 0.4534
[Stateless] Epoch 2/10 | Train Loss: 1.1527 | Train Acc: 0.5897
[Stateless] Epoch 3/10 | Train Loss: 0.9764 | Train Acc: 0.6535
[Stateless] Epoch 4/10 | Train Loss: 0.8500 | Train Acc: 0.6983
[Stateless] Epoch 5/10 | Train Loss: 0.7416 | Train Acc: 0.7379
[Stateless] Epoch 6/10 | Train Loss: 0.6345 | Train Acc: 0.7762
[Stateless] Epoch 7/10 | Train Loss: 0.5310 | Train Acc: 0.8127
[Stateless] Epoch 8/10 | Train Loss: 0.4342 | Train Acc: 0.8487
[Stateless] Epoch 9/10 | Train Loss: 0.3485 | Train Acc: 0.8807
[Stateless] Epoch 10/10 | Train Loss: 0.2653 | Train Acc: 0.9114
[Stateless] Combined Val Loss: 1.2115 | Combined Val Acc: 0.6787


In [20]:
# modelA was trained earlier on Subset A
# If needed, reload it:
# modelA = SimpleCNN().to(device)
# modelA.load_state_dict(torch.load("modelA_subsetA.pth"))
model_stateful = model  # reuse the same model
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_stateful.parameters(), lr=0.001)

num_epochs_stateful = 5

for epoch in range(num_epochs_stateful):
    model_stateful.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in subsetB_loader:
        images, labels = images.to(device), labels.to(device)
        # Remap labels from 5-9 to 0-4 for training
        labels = labels - 5

        optimizer.zero_grad()
        outputs = model_stateful(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(subsetB_loader)
    train_acc = correct / total

    print(f"[Stateful] Epoch {epoch+1}/{num_epochs_stateful} | "
          f"Train Loss (B): {train_loss:.4f} | Train Acc (B): {train_acc:.4f}")

# DataLoaders for evaluation on A and B
evalA_loader = DataLoader(subsetA, batch_size=64, shuffle=False)
evalB_loader = DataLoader(subsetB, batch_size=64, shuffle=False)

stateful_A_acc, stateful_A_loss = evaluate(model_stateful, evalA_loader, device)

# Manual evaluation for Subset B with label remapping, as evaluate() helper doesn't remap
model_stateful.eval()
correct_B, total_B, loss_sum_B = 0, 0, 0.0
criterion_eval = nn.CrossEntropyLoss()

with torch.no_grad():
    for images, labels in evalB_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels - 5 # Remap labels here for evaluation

        outputs = model_stateful(images)
        loss = criterion_eval(outputs, labels)
        loss_sum_B += loss.item()

        _, predicted = outputs.max(1)
        total_B += labels.size(0)
        correct_B += predicted.eq(labels).sum().item()

stateful_B_acc = correct_B / total_B
stateful_B_loss = loss_sum_B / len(evalB_loader)


print(f"[Stateful] Eval on Subset A (old classes): "
)

[Stateful] Epoch 1/5 | Train Loss (B): 2.2638 | Train Acc (B): 0.7225
[Stateful] Epoch 2/5 | Train Loss (B): 1.8359 | Train Acc (B): 0.8073
[Stateful] Epoch 3/5 | Train Loss (B): 1.9894 | Train Acc (B): 0.7532
[Stateful] Epoch 4/5 | Train Loss (B): 2.1832 | Train Acc (B): 0.7443
[Stateful] Epoch 5/5 | Train Loss (B): 1.3737 | Train Acc (B): 0.6995
[Stateful] Eval on Subset A (old classes): 


In [21]:
from torch.utils.data import random_split

# Split test set into two equal groups
test_size = len(test_data)
groupA_size = test_size // 2
groupB_size = test_size - groupA_size

groupA_data, groupB_data = random_split(test_data, [groupA_size, groupB_size])

groupA_loader = DataLoader(groupA_data, batch_size=64, shuffle=False)
groupB_loader = DataLoader(groupB_data, batch_size=64, shuffle=False)


In [24]:
# Evaluate stateless model on Group A
groupA_acc, groupA_loss = evaluate(model_stateless, groupA_loader, device)

# Evaluate stateful model on Group B with custom label remapping
model_stateful.eval()
correct_B_stateful, total_B_stateful, loss_sum_B_stateful = 0, 0, 0.0
criterion_eval = nn.CrossEntropyLoss()

with torch.no_grad():
    for images, labels in groupB_loader:
        images, labels = images.to(device), labels.to(device)

        # Custom label remapping for stateful model (5 classes).
        # It was trained on classes 0-4 and then fine-tuned on classes 5-9 (remapped to 0-4).
        # So, its 5 outputs are meant to categorize original {0,5} to 0, {1,6} to 1, etc.
        remapped_labels = torch.where(labels >= 5, labels - 5, labels)

        outputs = model_stateful(images)
        loss = criterion_eval(outputs, remapped_labels)
        loss_sum_B_stateful += loss.item()

        _, predicted = outputs.max(1)
        total_B_stateful += remapped_labels.size(0)
        correct_B_stateful += predicted.eq(remapped_labels).sum().item()

groupB_acc = correct_B_stateful / total_B_stateful
groupB_loss = loss_sum_B_stateful / len(groupB_loader)


print("=== A/B Test Results ===")
print(f"Group A (Stateless Model)  - Acc: {groupA_acc:.4f}, Loss: {groupA_loss:.4f}")
print(f"Group B (Stateful Model)   - Acc: {groupB_acc:.4f}, Loss: {groupB_loss:.4f}")

=== A/B Test Results ===
Group A (Stateless Model)  - Acc: 0.6732, Loss: 1.2462
Group B (Stateful Model)   - Acc: 0.1928, Loss: 11.4893


In [26]:
import random

epsilon = 0.1  # 10% exploration
model_rewards = [0, 0]  # total reward for each model
model_counts = [0, 0]   # number of times each model was chosen

models = [model_stateless, model_stateful]

def get_reward(model, image, label):
    """Reward = 1 if correct prediction, else 0."""
    model.eval()
    with torch.no_grad():
        output = model(image.to(device))
        _, pred = output.max(1)
        return 1 if pred.item() == label.item() else 0

# Use the full test set for simulation
test_loader_single = DataLoader(test_data, batch_size=1, shuffle=True)

num_rounds = 2000

for i, (image, label) in enumerate(test_loader_single):
    if i >= num_rounds:
        break

    # ε-greedy selection
    if random.random() < epsilon:
        chosen_model = random.choice([0, 1])  # explore
    else:
        chosen_model = 0 if model_rewards[0] >= model_rewards[1] else 1  # exploit

    # Get reward
    reward = get_reward(models[chosen_model], image, label)

    # Update stats
    model_rewards[chosen_model] += reward
    model_counts[chosen_model] += 1

print("=== Bandit Model Selection Results ===")
print(f"Stateless Model  - Chosen {model_counts[0]} times, Reward {model_rewards[0]}")
print(f"Stateful Model   - Chosen {model_counts[1]} times, Reward {model_rewards[1]}")

=== Bandit Model Selection Results ===
Stateless Model  - Chosen 1899 times, Reward 1278
Stateful Model   - Chosen 101 times, Reward 12
